<a href="https://colab.research.google.com/github/Riya-337/CervicalCancerDetection/blob/main/CervicalCancerDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install shap scikit-learn pandas numpy matplotlib ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
import shap
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
from datetime import datetime
!pip install flask pyngrok

# In-memory patient database (simulates hospital records)
patient_db = {}

# Clinical features (all continuous/%-based)
FEATURES = [
    "Age", "HPV_Viral_Load", "Cytology_Abnormality_Score",
    "Schiller_Test_Intensity", "Colposcopy_Suspicion_Score",
    "Time_Since_Last_Screen", "STD_Severity_Score"
]

In [ ]:
# Define feature order explicitly
FEATURES = [
    "Age", "HPV_Viral_Load", "Cytology_Abnormality_Score",
    "Schiller_Test_Intensity", "Colposcopy_Suspicion_Score",
    "Time_Since_Last_Screen", "STD_Severity_Score"
]

np.random.seed(42)
n = 2000

# Generate overlapping, realistic clinical data
X = pd.DataFrame({
    "Age": np.random.randint(18, 70, n),
    "HPV_Viral_Load": np.clip(np.random.exponential(25, n), 0, 100),
    "Cytology_Abnormality_Score": np.clip(np.random.beta(2, 4, n) * 100, 0, 100),
    "Schiller_Test_Intensity": np.clip(np.random.beta(1.5, 5, n) * 100, 0, 100),
    "Colposcopy_Suspicion_Score": np.clip(np.random.beta(2, 3, n) * 100, 0, 100),
    "Time_Since_Last_Screen": np.random.randint(0, 60, n),
    "STD_Severity_Score": np.clip(np.random.exponential(15, n), 0, 100),
})

# Realistic risk function with noise
risk_score = (
    0.25 * X["HPV_Viral_Load"] +
    0.20 * X["Cytology_Abnormality_Score"] +
    0.20 * X["Colposcopy_Suspicion_Score"] +
    0.10 * X["STD_Severity_Score"] +
    0.05 * X["Age"] +
    0.10 * X["Time_Since_Last_Screen"]
) / 100.0
risk_score = np.clip(risk_score + np.random.normal(0, 0.1, n), 0, 1)

y = (risk_score > 0.5).astype(int)

# Enforce feature order
X = X[FEATURES]

# Train base model
base_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
base_model.fit(X, y)

# Calibrate for smooth probabilities
calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
calibrated_model.fit(X, y)

# Use calibrated for prediction, base for SHAP
model = calibrated_model
explainer = shap.TreeExplainer(base_model)

print("✅ Model trained with calibrated probabilities and SHAP support")
print("Sample predictions:", model.predict_proba(X.head())[:, 1].round(2))

NameError: name 'CalibratedClassifierCV' is not defined

In [ ]:
!pip install flask pyngrok

from flask import Flask, request, jsonify
import threading
from pyngrok import ngrok

# >>> STEP 1: SET YOUR NGROK AUTH TOKEN HERE <<<
ngrok.set_auth_token("37OfXprJPpmRUi97c8ID1OiC7pq_7YU2hQeh18Bnnpms9tHXd")

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict_api():
    try:
        data = request.json
        x_input = pd.DataFrame([data], columns=FEATURES)
        proba = model.predict_proba(x_input)[0]
        risk = float(proba[1])
        pred = "Malignant" if risk > 0.5 else "Benign"

        shap_vals = explainer.shap_values(x_input)[0]
        top3 = sorted(zip(FEATURES, shap_vals), key=lambda x: abs(x[1]), reverse=True)[:3]
        reason = ", ".join([f"{k.replace('_', ' ')} ↑" if v > 0 else f"{k.replace('_', ' ')} ↓" for k, v in top3])

        return jsonify({
            "prediction": pred,
            "risk_score_percent": round(risk * 100, 1),
            "reason": reason
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

# Run Flask on port 5001
def run_flask():
    app.run(host='0.0.0.0', port=5001)

threading.Thread(target=run_flask, daemon=True).start()
print("✅ Flask API running on port 5001")

# Expose via ngrok
public_url = ngrok.connect(5001)
print(f"\n🌐 Your public URL: {public_url}/predict")

In [ ]:
# In-memory patient database
patient_db = {}

def run_ai_audit(patient_id, data_dict):
    if patient_id not in patient_db:
        patient_db[patient_id] = []

    # Use DataFrame with exact feature order
    x_input = pd.DataFrame([data_dict], columns=FEATURES)

    # Get calibrated probability
    proba = model.predict_proba(x_input)[0]
    risk_score = float(proba[1])
    prediction = "Malignant" if risk_score > 0.5 else "Benign"

    # SHAP explanation (using base model)
    shap_vals = explainer.shap_values(x_input)[0]
    top3 = sorted(zip(FEATURES, shap_vals), key=lambda x: abs(x[1]), reverse=True)[:3]
    reason = ", ".join([f"{k.replace('_', ' ')} ↑" if v > 0 else f"{k.replace('_', ' ')} ↓" for k, v in top3])

    record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "data": data_dict,
        "risk_score": risk_score,
        "prediction": prediction,
        "reason": reason
    }
    patient_db[patient_id].append(record)
    return record

In [ ]:
# Build input form
patient_id_input = widgets.Text(value='PRIYA_001', description='Patient ID:')
age_input = widgets.IntSlider(value=34, min=18, max=80, description='Age:')
hpv_input = widgets.FloatSlider(value=85, min=0, max=100, step=1, description='HPV Viral Load (%):')
cytology_input = widgets.FloatSlider(value=88, min=0, max=100, step=1, description='Cytology Abnormality (%):')
colpo_input = widgets.FloatSlider(value=90, min=0, max=100, step=1, description='Colposcopy Suspicion (%):')
schiller_input = widgets.FloatSlider(value=60, min=0, max=100, step=1, description='Schiller Test (%):')
std_input = widgets.FloatSlider(value=40, min=0, max=100, step=1, description='STD Severity (%):')
time_input = widgets.IntSlider(value=18, min=0, max=60, description='Months Since Last Screen:')

output_area = widgets.Output()

def on_submit_click(b):
    data = {
        "Age": age_input.value,
        "HPV_Viral_Load": hpv_input.value,
        "Cytology_Abnormality_Score": cytology_input.value,
        "Schiller_Test_Intensity": schiller_input.value,
        "Colposcopy_Suspicion_Score": colpo_input.value,
        "Time_Since_Last_Screen": time_input.value,
        "STD_Severity_Score": std_input.value
    }
    x_input = pd.DataFrame([data], columns=FEATURES)
    result = run_ai_audit(patient_id_input.value, data)

    color = "#d32f2f" if result["prediction"] == "Malignant" else "#388e3c"
    with output_area:
        output_area.clear_output()
        # ✅ FIXED: This block is NOW inside 'with output_area:'
        display(HTML(f"""
        <div style="padding:15px; border-left:4px solid {color}; background:#f9f9f9; margin-top:10px; color:#111; font-family:Arial, sans-serif;">
            <h3 style="color:#111; margin-top:0;">🩺 AI Audit Result for {result['timestamp']}</h3>
            <p><strong>Prediction:</strong> <span style="color:{color}; font-weight:bold;">{result['prediction']}</span></p>
            <p><strong>Risk Score:</strong> <span style="color:#111;">{result['risk_score']*100:.1f}%</span></p>
            <p><strong>Key Drivers:</strong> <span style="color:#111;">{result['reason']}</span></p>
            <p style="color:#c62828; font-weight:bold; margin-bottom:0; background:#ffebee; padding:8px; border-radius:4px;">
                ⚠️ This is a second opinion. A clinician must review before taking clinical action.
            </p>
        </div>
        """))

        # Plot SHAP
        shap_values = explainer(x_input)
        plt.figure(figsize=(8, 3))
        shap.plots.waterfall(shap_values[0], max_display=7, show=False)
        plt.tight_layout()
        plt.show()

submit_btn = widgets.Button(description="🔍 Run AI Audit", button_style='success')
submit_btn.on_click(on_submit_click)

ui = widgets.VBox([
    widgets.HTML("<h2>🔬 Add New Patient Visit</h2>"),
    patient_id_input,
    age_input,
    hpv_input,
    cytology_input,
    colpo_input,
    schiller_input,
    std_input,
    time_input,
    submit_btn,
    output_area
])

display(ui)

In [ ]:
# Show ranked patient list
def show_triage():
    all_patients = []
    for pid, visits in patient_db.items():
        latest = max(visits, key=lambda x: x["risk_score"])
        all_patients.append({
            "Patient ID": pid,
            "Max Risk (%)": round(latest["risk_score"] * 100, 1),
            "Prediction": latest["prediction"],
            "Last Visit": latest["timestamp"]
        })
    if not all_patients:
        print("📭 No patients recorded yet.")
        return
    df = pd.DataFrame(all_patients).sort_values("Max Risk (%)", ascending=False)
    display(HTML("<h3>🚨 Urgent Triage List (Top 10)</h3>"))
    display(df.head(10).style.set_properties(**{'text-align': 'left'}))

show_triage()

In [ ]:
# Get API URL (from user input)
print("Paste your ngrok API URL from above (e.g., https://abc123.ngrok.app):")
API_BASE = input().strip().rstrip('/')

# Generate HTML with properly escaped JS
mobile_html = f'''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="hood" content="width=device-width, initial-scale=1.0"/>
  <title>🩺 Cervical AI Audit</title>
  <style>
    body {{ font-family: Arial, sans-serif; background: #f9f9f9; padding: 15px; color: #111; }}
    h1 {{ color: #1b5e20; text-align: center; }}
    .form {{ background: white; padding: 15px; border-radius: 10px; box-shadow: 0 2px 5px #ccc; }}
    .field {{ margin: 12px 0; }}
    label {{ display: block; font-weight: bold; margin-bottom: 5px; color: #222; }}
    input {{ width: 100%; padding: 10px; border: 1px solid #ccc; border-radius: 6px; font-size: 16px; }}
    button {{
      width: 100%; background: #2E8B57; color: white; padding: 14px;
      border: none; border-radius: 8px; font-size: 18px; font-weight: bold;
    }}
    #result {{
      margin-top: 20px; padding: 15px; border-radius: 8px; display: none;
      background: white; box-shadow: 0 1px 3px rgba(0,0,0,0.1);
    }}
    .high {{ border-left: 4px solid #d32f2f; }}
    .low {{ border-left: 4px solid #388e3c; }}
    .loading {{ color: #1976d2; }}
  </style>
</head>
<body>
  <h1>🔍 Cervical Cancer AI Audit</h1>
  <div class="form">
    <div class="field"><label>Age</label><input type="number" id="Age" value="34" min="18" max="80"></div>
    <div class="field"><label>HPV Viral Load (%)</label><input type="number" id="HPV_Viral_Load" value="85" min="0" max="100" step="0.1"></div>
    <div class="field"><label>Cytology Abnormality (%)</label><input type="number" id="Cytology_Abnormality_Score" value="88" min="0" max="100" step="0.1"></div>
    <div class="field"><label>Colposcopy Suspicion (%)</label><input type="number" id="Colposcopy_Suspicion_Score" value="90" min="0" max="100" step="0.1"></div>
    <div class="field"><label>Schiller Test (%)</label><input type="number" id="Schiller_Test_Intensity" value="60" min="0" max="100" step="0.1"></div>
    <div class="field"><label>STD Severity (%)</label><input type="number" id="STD_Severity_Score" value="40" min="0" max="100" step="0.1"></div>
    <div class="field"><label>Months Since Last Screen</label><input type="number" id="Time_Since_Last_Screen" value="18" min="0" max="60"></div>

    <button onclick="analyze()">🩺 Analyze Risk</button>
    <div id="result"></div>
  </div>

  <script>
    const API_URL = "{API_BASE}/predict";

    async function analyze() {{
      const resultDiv = document.getElementById("result");
      resultDiv.innerHTML = '<p class="loading">⏳ Analyzing with AI...</p>';
      resultDiv.style.display = "block";

      const data = {{
        Age: parseFloat(document.getElementById("Age").value),
        HPV_Viral_Load: parseFloat(document.getElementById("HPV_Viral_Load").value),
        Cytology_Abnormality_Score: parseFloat(document.getElementById("Cytology_Abnormality_Score").value),
        Schiller_Test_Intensity: parseFloat(document.getElementById("Schiller_Test_Intensity").value),
        Colposcopy_Suspicion_Score: parseFloat(document.getElementById("Colposcopy_Suspicion_Score").value),
        Time_Since_Last_Screen: parseFloat(document.getElementById("Time_Since_Last_Screen").value),
        STD_Severity_Score: parseFloat(document.getElementById("STD_Severity_Score").value)
      }};

      try {{
        const res = await fetch(API_URL, {{
          method: "POST",
          headers: {{ "Content-Type": "application/json" }},
          body: JSON.stringify(data)
        }});
        const r = await res.json();

        if (r.error) {{
          // ✅ CORRECT: Use double braces to output literal {{ and }} in JS
          resultDiv.innerHTML = `<p style="color:#d32f2f;">❌ Error: ${{r.error}}</p>`;
          return;
        }}

        const isHigh = r.prediction === "Malignant";
        resultDiv.className = isHigh ? "high" : "low";
        resultDiv.innerHTML = `
          <h3>${{r.prediction}} (${{r.risk_score_percent}}%)</h3>
          <p><strong>Key Drivers:</strong> ${{r.reason}}</p>
          <p style="color:#c62828; font-weight:bold; margin:10px 0 0 0;">
            ⚠️ AI audit only. Doctor must review before action.
          </p>
        `;
      }} catch (e) {{
        resultDiv.innerHTML = `<p style="color:#d32f2f;">❌ Failed to connect to AI server.<br>Make sure Colab is running!</p>`;
      }}
    }}
  </script>
</body>
</html>
'''

# Save file
with open("/content/cervical_mobile_live.html", "w") as f:
    f.write(mobile_html)

print("\n✅ Live mobile app saved! Download 'cervical_mobile_live.html' and open on your phone.")